In [36]:
import pandas as pd
from sqlalchemy import create_engine
!pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable


In [37]:
CSV_PATH="E-commerce_Customer_Segmentation_2026.csv"

In [38]:
DB_CONFIG={
    "user":"root",
    "password":"12345",
    "host":"localhost",
    "database":"ecommerce_segmentation"
}

In [39]:
df = pd.read_csv(CSV_PATH)
 
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)


Shape: (50000, 53)

Dtypes:
 customer_id                       object
customer_segment                  object
segment_category                  object
age                                int64
age_group                         object
gender                            object
country                           object
city                              object
income_bracket                    object
education_level                   object
employment_type                   object
marital_status                    object
tenure_months                      int64
total_purchases                    int64
avg_order_value_usd              float64
total_spent_usd                  float64
purchase_frequency                object
days_since_last_purchase           int64
preferred_category_1              object
preferred_category_2              object
preferred_category_3              object
shopping_channel                  object
device_used                       object
payment_method              

In [40]:
print("\nNull counts:\n", df.isnull().sum())



Null counts:
 customer_id                          0
customer_segment                     0
segment_category                     0
age                                  0
age_group                            0
gender                               0
country                              0
city                                 0
income_bracket                       0
education_level                      0
employment_type                      0
marital_status                       0
tenure_months                        0
total_purchases                      0
avg_order_value_usd                  0
total_spent_usd                      0
purchase_frequency                   0
days_since_last_purchase             0
preferred_category_1                 0
preferred_category_2             16743
preferred_category_3             33257
shopping_channel                     0
device_used                          0
payment_method                       0
return_count                         0
complaint_

In [41]:
print("\nSample rows:\n", df.head())


Sample rows:
    customer_id customer_segment     segment_category  age age_group  gender  \
0  CUST-000001         Consumer  High Value Inactive   70       65+    Male   
1  CUST-000002          Premium  High Value Inactive   53     45-54  Female   
2  CUST-000003         Consumer  High Value Inactive   82       65+    Male   
3  CUST-000004       Enterprise    High Value Active   18     18-24    Male   
4  CUST-000005          Premium    High Value Active   64     55-64    Male   

                country       city income_bracket education_level  ...  \
0                Canada   Winnipeg          150K+    Professional  ...   
1               Germany      Essen        25K-50K     High School  ...   
2                Mexico     Puebla       75K-100K       Bachelors  ...   
3             Singapore  Singapore          150K+         Masters  ...   
4  United Arab Emirates   Fujairah          150K+             PhD  ...   

  customer_value_category activity_status  health_status  churn_r

In [42]:
# clean the column

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]", "", regex=True)
)

In [43]:
print(df.columns)

Index(['customer_id', 'customer_segment', 'segment_category', 'age',
       'age_group', 'gender', 'country', 'city', 'income_bracket',
       'education_level', 'employment_type', 'marital_status', 'tenure_months',
       'total_purchases', 'avg_order_value_usd', 'total_spent_usd',
       'purchase_frequency', 'days_since_last_purchase',
       'preferred_category_1', 'preferred_category_2', 'preferred_category_3',
       'shopping_channel', 'device_used', 'payment_method', 'return_count',
       'complaint_count', 'satisfaction_score', 'satisfaction_level',
       'loyalty_tier', 'email_open_rate', 'click_through_rate',
       'conversion_rate', 'social_media_presence',
       'customer_lifetime_value_usd', 'customer_acquisition_cost_usd',
       'customer_profitability_usd', 'recency_score', 'frequency_score',
       'monetary_score', 'rfm_score', 'churn_risk_score',
       'customer_health_score', 'dataset_year', 'customer_value_category',
       'activity_status', 'health_status',

In [44]:
# Drop exact duplicate rows 
before = len(df)
df = df.drop_duplicates()
print(f"\nDropped {before - len(df)} duplicate rows")


Dropped 0 duplicate rows


In [45]:
# Trim whitespace on text/object columns
obj_cols = df.select_dtypes(include="object").columns
for col in obj_cols:
    df[col] = df[col].str.strip()

In [46]:
print(obj_cols)

Index(['customer_id', 'customer_segment', 'segment_category', 'age_group',
       'gender', 'country', 'city', 'income_bracket', 'education_level',
       'employment_type', 'marital_status', 'purchase_frequency',
       'preferred_category_1', 'preferred_category_2', 'preferred_category_3',
       'shopping_channel', 'device_used', 'payment_method',
       'satisfaction_level', 'loyalty_tier', 'social_media_presence',
       'customer_value_category', 'activity_status', 'health_status',
       'churn_risk_category', 'rfm_category', 'profitability_category',
       'engagement_level', 'behavior_segment', 'device_preference',
       'clv_category'],
      dtype='object')


In [50]:
conn_str = (
    f"mysql+mysqlconnector://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}/{DB_CONFIG['database']}"
)
engine = create_engine(conn_str)
 
df.to_sql("staging_customers", engine, if_exists="replace", index=False, chunksize=5000)
 
print(f"\nLoaded {len(df)} rows into staging_customers")


Loaded 50000 rows into staging_customers
